# 25 - Diffusion Policy Intuition

## What / Why / How

**What we are trying to do:** Develop intuition for diffusion-style action sampling on a multimodal action problem.

**Why this matters:** Manipulation often has multiple valid futures. Predicting the average action can be physically wrong or unsafe.

**How we will do it:** Create two action modes, train a tiny local denoiser, sample actions, and compare sampled modes against the bad mean.

## Goal

Build intuition for diffusion policies.

This is not a production diffusion model. It is a tiny denoising toy that shows why sampling can preserve multiple action modes where mean regression fails.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
rng = np.random.default_rng(25)
n = 600
modes = rng.choice([-1.0, 1.0], size=n)
clean_actions = np.c_[0.8 + rng.normal(0, 0.04, n), modes * 0.55 + rng.normal(0, 0.05, n)]
sigma = 0.35
noisy = clean_actions + rng.normal(0, sigma, size=clean_actions.shape)
print("clean mean:", clean_actions.mean(axis=0))
print("mean action would hit obstacle:", abs(clean_actions.mean(axis=0)[1]) < 0.2)

In [ ]:
# Train a local denoiser: predict clean action as weighted average of nearest noisy examples.
def denoise(x, bandwidth=0.35):
    distances = np.linalg.norm(noisy - x[None, :], axis=1)
    weights = np.exp(-0.5 * (distances / bandwidth) ** 2)
    weights = weights / (weights.sum() + 1e-9)
    return weights @ clean_actions

samples = rng.normal(0, 1.0, size=(60, 2))
for step in range(12):
    for i in range(len(samples)):
        target = denoise(samples[i])
        samples[i] = 0.75 * samples[i] + 0.25 * target + rng.normal(0, 0.02, 2)

print("sample mean:", samples.mean(axis=0))
print("sample y modes approx:", np.percentile(samples[:, 1], [10, 50, 90]))

In [ ]:
safe_samples = samples[np.abs(samples[:, 1]) > 0.2]
chosen = safe_samples[0]
print("chosen sampled action:", chosen)

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(5, 4))
    plt.scatter(clean_actions[:, 0], clean_actions[:, 1], s=8, alpha=0.2, label="data")
    plt.scatter(samples[:, 0], samples[:, 1], s=20, label="samples")
    plt.axhspan(-0.2, 0.2, color="black", alpha=0.1, label="bad mean zone")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Add a third action mode.
2. Reduce the number of demonstrations.
3. Explain how action chunks differ from single-step actions.